# Weather Agent — Challenge 1


## Setup

Run each cell in order. In Colab Enterprise, install `google-adk` in a separate code cell if needed (`%pip install google-adk`). Vertex AI uses your current student project.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any
import math
import os

print("Python standard-library weather simulation loaded.")

Python standard-library weather simulation loaded.


In [2]:
import google.auth

credentials, project_id = google.auth.default()

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"

print("Vertex AI configured")
print("Project:", project_id)
print("Location:", os.environ["GOOGLE_CLOUD_LOCATION"])

Vertex AI configured
Project: qwiklabs-gcp-00-2d0126732d9c
Location: global


## City weather data

Five cities with sample weather conditions, adding more apabilities as I continue.


In [3]:
@dataclass(frozen=True)
class WeatherFixture:
    city: str
    state: str
    latitude: float
    longitude: float
    condition: str
    temperature_f: int
    wind_mph: int
    precipitation_probability_pct: int


FIXTURES: tuple[WeatherFixture, ...] = (
    WeatherFixture("Miami", "FL", 25.7617, -80.1918, "Sunny", 105, 8, 10),
    WeatherFixture("Denver", "CO", 39.7392, -104.9903, "Heavy snow", 18, 22, 90),
    WeatherFixture("Seattle", "WA", 47.6062, -122.3321, "Light rain", 56, 7, 75),
    WeatherFixture("Phoenix", "AZ", 33.4484, -112.0740, "Sunny", 116, 5, 0),
    WeatherFixture("New York", "NY", 40.7128, -74.0060, "Severe thunderstorms", 76, 38, 95),
)


def _city_key(name: str) -> str:
    """Normalize a user's city spelling for lookup."""
    return " ".join(name.casefold().strip().replace(",", " ").split())


CITY_INDEX: dict[str, WeatherFixture] = {}
for fixture in FIXTURES:
    CITY_INDEX[_city_key(fixture.city)] = fixture
    CITY_INDEX[_city_key(f"{fixture.city}, {fixture.state}")] = fixture


def resolve_city(city: str) -> dict[str, Any]:
    """Resolve one of five supported US cities to its simulated city-center coordinates.

    Args:
        city: City name, optionally followed by its state abbreviation.

    Returns:
        A status and city/state/latitude/longitude, or an informative error.
    """
    fixture = CITY_INDEX.get(_city_key(city))
    if fixture is None:
        return {"status": "error", "message": "Unsupported city; choose Miami, Denver, Seattle, Phoenix, or New York."}
    return {"status": "success", "city": fixture.city, "state": fixture.state,
            "latitude": fixture.latitude, "longitude": fixture.longitude}


print("Supported cities:", ", ".join(f.city for f in FIXTURES))

Supported cities: Miami, Denver, Seattle, Phoenix, New York


## Weather tools


In [4]:
def get_weather_at_coordinates(latitude: float, longitude: float) -> dict[str, Any]:
    """Get simulated weather for one of five configured US city coordinates.

    Args:
        latitude: Decimal latitude returned by the city-lookup tool.
        longitude: Decimal longitude returned by the city-lookup tool.

    Returns:
        A structured simulated observation or an error when coordinates are unsupported.
    """
    if not (math.isfinite(latitude) and math.isfinite(longitude)):
        return {"status": "error", "message": "Coordinates must be finite numbers."}
    for fixture in FIXTURES:
        if abs(latitude - fixture.latitude) < 0.00001 and abs(longitude - fixture.longitude) < 0.00001:
            return {
                "status": "success", "city": fixture.city, "state": fixture.state,
                "latitude": fixture.latitude, "longitude": fixture.longitude,
                "condition": fixture.condition, "temperature_f": fixture.temperature_f,
                "wind_mph": fixture.wind_mph,
                "precipitation_probability_pct": fixture.precipitation_probability_pct,
                "source": "sample_weather_data", "is_live": False,
            }
    return {"status": "error", "message": "No simulated weather is configured for these coordinates."}


def classify_simulated_weather(observation: dict[str, Any]) -> dict[str, str]:
    """Apply deterministic training-only alert rules to an observation.

    Args:
        observation: A successfully retrieved simulated weather result.

    Returns:
        Alert severity, title, and an explanatory message.
    """
    if observation.get("status") != "success":
        return {"severity": "error", "title": "Weather unavailable", "message": "No valid simulated observation."}
    condition = str(observation["condition"]).casefold()
    temp = int(observation["temperature_f"])
    wind = int(observation["wind_mph"])
    if "thunderstorm" in condition or wind >= 35:
        return {"severity": "severe", "title": "Simulated severe-weather alert",
                "message": "Thunderstorms or damaging winds appear in the training scenario."}
    if "snow" in condition:
        return {"severity": "warning", "title": "Simulated winter-weather alert",
                "message": "Snow conditions appear in the training scenario."}
    if temp >= 110:
        return {"severity": "warning", "title": "Simulated extreme-heat warning",
                "message": "The fixture temperature meets the training threshold of 110 F."}
    if temp >= 100:
        return {"severity": "advisory", "title": "Simulated heat advisory",
                "message": "The fixture temperature meets the training threshold of 100 F."}
    return {"severity": "normal", "title": "Routine weather summary",
            "message": "No training alert threshold was reached."}


def weather_summary(city: str) -> str:
    """Provide a transparent sample summary using the two custom weather tools.

    Args:
        city: Supported U.S. city name.

    Returns:
        A concise simulated weather report or an unsupported-city error.
    """
    location = resolve_city(city)
    if location["status"] != "success":
        return location["message"]
    weather = get_weather_at_coordinates(location["latitude"], location["longitude"])
    if weather["status"] != "success":
        return weather["message"]
    alert = classify_simulated_weather(weather)
    return (f"SIMULATED — {weather['city']}, {weather['state']}: "
            f"{weather['condition']}, {weather['temperature_f']} F, "
            f"wind {weather['wind_mph']} mph. "
            f"{alert['title']}: {alert['message']} "
            "Not live weather or an official NWS warning.")

In [5]:
import requests

def get_nws_weather(latitude: float, longitude: float) -> dict:
    """Retrieve a U.S. weather forecast using the NWS API."""

    if not (-90 <= latitude <= 90 and -180 <= longitude <= 180):
        return {"error": "Invalid coordinates"}

    headers = {
        "User-Agent": "WeatherAgent/1.0 (workshop educational demo)",
        "Accept": "application/geo+json"
    }

    try:
        response = requests.get(
            f"https://api.weather.gov/points/{latitude},{longitude}",
            headers=headers,
            timeout=10
        )

        if response.status_code == 404:
            return {
                "error": "Location is outside NWS forecast coverage."
            }

        response.raise_for_status()

        forecast_url = response.json()["properties"]["forecast"]

        forecast = requests.get(
            forecast_url,
            headers=headers,
            timeout=10
        )
        forecast.raise_for_status()

        period = forecast.json()["properties"]["periods"][0]

        return {
            "source": "National Weather Service",
            "forecast": period["shortForecast"],
            "temperature": period["temperature"],
            "details": period["detailedForecast"]
        }

    except (requests.RequestException, KeyError, IndexError) as error:
        return {"error": str(error)}


def geocode_location(city: str) -> dict:
    """Find coordinates for cities worldwide using Open-Meteo."""
    try:
        response = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 5},
            timeout=10
        )
        response.raise_for_status()

        results = response.json().get("results", [])

        if not results:
            return {"error": "Location not found"}

        locations = [
            {
                "city": place["name"],
                "country": place.get("country", ""),
                "region": place.get("admin1", ""),
                "latitude": place["latitude"],
                "longitude": place["longitude"]
            }
            for place in results
        ]

        return {"locations": locations}

    except requests.RequestException as error:
        return {"error": str(error)}


def get_open_meteo_weather(latitude: float, longitude: float) -> dict:
    """Retrieve current weather from Open-Meteo."""
    try:
        response = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": latitude,
                "longitude": longitude,
                "current": "temperature_2m,wind_speed_10m",
                "temperature_unit": "fahrenheit",
                "wind_speed_unit": "mph"
            },
            timeout=10
        )
        response.raise_for_status()
        return response.json()["current"]
    except Exception as error:
        return {"error": str(error)}

In [6]:
location = geocode_location("Miami")
print("Geocoding results:", location)

if "error" not in location and location["locations"]:
    miami = next(
        (
            place for place in location["locations"]
            if place["city"] == "Miami"
            and place["country"] == "United States"
            and place["region"] == "Florida"
        ),
        None
    )

    if miami:
        lat = miami["latitude"]
        lon = miami["longitude"]

        print("NWS:", get_nws_weather(lat, lon))
        print("Open-Meteo:", get_open_meteo_weather(lat, lon))
    else:
        print("Miami, Florida was not found.")
else:
    print("Location lookup failed:", location.get("error"))

Geocoding results: {'locations': [{'city': 'Miami', 'country': 'United States', 'region': 'Florida', 'latitude': 25.77427, 'longitude': -80.19366}, {'city': 'Miami', 'country': 'United States', 'region': 'Oklahoma', 'latitude': 36.87451, 'longitude': -94.87746}, {'city': 'Miami', 'country': 'United States', 'region': 'Texas', 'latitude': 35.69143, 'longitude': -100.63819}, {'city': 'Miami', 'country': 'United States', 'region': 'Arizona', 'latitude': 33.39922, 'longitude': -110.86872}, {'city': 'Miamitown', 'country': 'United States', 'region': 'Ohio', 'latitude': 39.21589, 'longitude': -84.70411}]}
NWS: {'source': 'National Weather Service', 'forecast': 'Showers And Thunderstorms Likely', 'temperature': 81, 'details': 'Showers and thunderstorms likely. Partly sunny, with a high near 81. Northeast wind around 29 mph, with gusts as high as 38 mph. Chance of precipitation is 60%. New rainfall amounts less than a tenth of an inch possible.'}
Open-Meteo: {'time': '2026-10-26T21:15', 'inter

## Weather examples


In [7]:
for fixture in FIXTURES:
    print(weather_summary(fixture.city))

SIMULATED — Miami, FL: Sunny, 105 F, wind 8 mph. Simulated heat advisory: The fixture temperature meets the training threshold of 100 F. Not live weather or an official NWS warning.
SIMULATED — Denver, CO: Heavy snow, 18 F, wind 22 mph. Simulated winter-weather alert: Snow conditions appear in the training scenario. Not live weather or an official NWS warning.
SIMULATED — Seattle, WA: Light rain, 56 F, wind 7 mph. Routine weather summary: No training alert threshold was reached. Not live weather or an official NWS warning.
SIMULATED — Phoenix, AZ: Sunny, 116 F, wind 5 mph. Simulated extreme-heat warning: The fixture temperature meets the training threshold of 110 F. Not live weather or an official NWS warning.
SIMULATED — New York, NY: Severe thunderstorms, 76 F, wind 38 mph. Simulated severe-weather alert: Thunderstorms or damaging winds appear in the training scenario. Not live weather or an official NWS warning.


## Tests

Offline five-city tests do not require Gemini or internet access.

In [8]:
def run_offline_tests() -> None:
    """Execute deterministic assertions without ADK or external network access."""
    expected = {"Miami": "advisory", "Denver": "warning", "Seattle": "normal",
                "Phoenix": "warning", "New York": "severe"}
    for city, severity in expected.items():
        loc = resolve_city(city)
        assert loc["status"] == "success", city
        obs = get_weather_at_coordinates(loc["latitude"], loc["longitude"])
        assert obs["status"] == "success" and obs["city"] == city, city
        assert obs["is_live"] is False and obs["source"] == "sample_weather_data"
        assert classify_simulated_weather(obs)["severity"] == severity, city
        assert "SIMULATED" in weather_summary(city)
        print(f"PASS — {city}: {severity}")
    assert resolve_city("miami, fl")["city"] == "Miami"
    assert resolve_city("  NEW   YORK  ")["city"] == "New York"
    assert resolve_city("Boston")["status"] == "error"
    assert get_weather_at_coordinates(0.0, 0.0)["status"] == "error"
    assert get_weather_at_coordinates(float("nan"), 0.0)["status"] == "error"
    assert "Unsupported city" in weather_summary("Boston")
    print("PASS — normalization, unsupported city, invalid coordinates, synthetic-data labeling")
    print("ALL OFFLINE TESTS PASSED")


run_offline_tests()

PASS — Miami: advisory
PASS — Denver: warning
PASS — Seattle: normal
PASS — Phoenix: warning
PASS — New York: severe
PASS — normalization, unsupported city, invalid coordinates, synthetic-data labeling
ALL OFFLINE TESTS PASSED


## Google ADK agent

This agent uses the live weather tools. It does not use the simulated five-city tools for live responses.

In [9]:
from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner
from google.genai import types

ADK_AVAILABLE = True

root_agent = LlmAgent(
    name="weather_alert_agent",

    model="gemini-3.5-flash-lite",

    description="Retrieves live weather for locations using NWS and Open-Meteo.",

    instruction=(
        "You are a friendly weather assistant. "
        "Only answer weather-related questions. "
        "Respond to weather requests for locations worldwide. "
        "Use geocode_location to resolve city coordinates. "
        "Use get_open_meteo_weather for current worldwide weather. "
        "Use get_nws_weather only for locations in the United States. "
        "If geocoding returns multiple locations, choose only when "
        "the user's city and region identify a clear match; "
        "otherwise ask the user to clarify before checking weather. "
        "If an API fails, explain that weather data is unavailable. "
        "Never invent weather conditions or official weather alerts. "
        "Do not present sample weather as live weather."
    ),

    tools=[
        geocode_location,
        get_open_meteo_weather,
        get_nws_weather
    ],

)

print("ADK agent ready:", root_agent.name)

ADK agent ready: weather_alert_agent


## Live agent tests

These tests require Gemini access through Vertex AI and internet access for the weather providers.

In [10]:
async def ask_agent(question: str) -> str:
    """Send a question to the weather agent."""

    runner = InMemoryRunner(
        agent=root_agent,
        app_name="weather_agent"
    )

    import uuid

    user_id = "workshop_user"
    session_id = str(uuid.uuid4())

    await runner.session_service.create_session(
        app_name="weather_agent",
        user_id=user_id,
        session_id=session_id
    )

    message = types.Content(
        role="user",
        parts=[types.Part(text=question)]
    )

    replies = []

    async for event in runner.run_async(
        user_id=user_id,
        session_id=session_id,
        new_message=message
    ):
        if event.is_final_response() and event.content:
            replies.extend(
                part.text
                for part in (event.content.parts or [])
                if getattr(part, "text", None)
            )

    return "\n".join(replies)


async def test_weather_agent():
    print("TEST 1 — Weather request")
    print(await ask_agent("What's the weather in Miami?"))

    print("\nTEST 2 — Non-U.S. location blocked")
    print(await ask_agent("What's the weather in Tokyo?"))

    print("\nTEST 3 — Unrelated request blocked")
    print(await ask_agent("Where can I buy a laptop?"))


await test_weather_agent()

TEST 1 — Weather request


/usr/local/lib/python3.12/dist-packages/google/adk/models/llm_request.py:273: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()


The current weather in Miami, Florida is 81°F with showers and thunderstorms likely, partly sunny skies, and northeast winds around 29 mph with gusts up to 38 mph. The chance of precipitation is 60%.

TEST 2 — Non-U.S. location blocked
The current weather in Tokyo, Japan is 60.6°F with a wind speed of 2.9 mph.

TEST 3 — Unrelated request blocked
I'm a weather assistant, so I can only help you with weather-related questions! If you'd like to check the forecast before heading out to shop for a laptop, just let me know the city you're in.
